# Proposal: Student Reading Support System

This notebook presents a concise proposal for a system to help school districts increase student reading by augmenting librarian recommendations with data-driven suggestions and LLM-powered pedagogical explanations. It includes: the use case, architecture diagrams, rollout and timeline estimates, risks & mitigations, and a runnable proof-of-concept recommender using TF-IDF over a librarian-provided catalog and borrow history.

## Use case (provided)

You are an engineering leader preparing a proposal for a pro-bono project: help a school district increase books read by students. Librarians reported that students follow librarian recommendations and librarians have anecdotal evidence about 
. Librarians can provide:
- historical checkout logs (who borrowed which book and when),
- the electronic catalog (metadata: title, author, subject, short description),

Goal: design and prototype a system that surfaces better, personalized recommendations to students and gives librarians data and tooling to spot trends and run targeted campaigns (e.g., reading challenges).

## High-level solution overview

Core components:
- Data ingestion: pipelines to import library checkout history and the electronic catalog (CSV, library management system export).
- Feature store: precomputed features for books (text embeddings, topics, age range) and student profiles (recent borrow history, inferred interests).
- Recommender service: candidate retrieval (content-based TF-IDF or embedding search), ranking (personalization), and freshness filters (recently added / recommended by classmates).
- Librarian console: UI for viewing cohort-level trends, approving campaigns, and providing feedback to the recommender (implicit/expert signals).
- Student UI: lightweight web app or LMS plugin that shows curated, age-appropriate suggestions and short LLM-generated blurbs that explain why a student might enjoy a book.
- Monitoring & safety: usage metrics, diversity/fairness checks, and prompt-safety guards before LLM-generated messaging is shown to students.

LLMs are used for: generating friendly explanatory blurbs, crafting reading-challenge text, and synthesizing librarian feedback into improvement signals for ranking. Embeddings (controlled, private) can be used for semantic similarity and clustering.

## Architecture diagram (conceptual)

```mermaid
flowchart LR
  Catalog((Electronic Catalog)) -->|ingest| ETL[ETL & Features]
  Checkout((Borrow History)) -->|ingest| ETL
  ETL --> FeatureStore[(Feature Store / Vector DB)]
  FeatureStore --> Recommender[Recommender Service]
  Recommender --> StudentUI[Student App / LMS Plugin]
  Recommender --> LibrarianConsole[Librarian Console]
  LLM[Hermes3 / LLM] -->|explain| StudentUI
  LibrarianConsole -->|feedback| Recommender

In [2]:
## Rollout plan & timeline (high-level)

1) Discovery & data access (2-3 weeks):
   - Confirm data exports and privacy constraints; obtain sample catalog & 3 months of anonymized checkout logs.

2) Prototype & POC (4-6 weeks):
   - Build ingestion scripts, a TF-IDF/content-based recommender, and a simple web UI for librarians to preview recommendations. Produce a short report and demo.

3) Pilot (8-12 weeks):
   - Deploy to 1-3 schools, collect engagement metrics, gather librarian feedback, and iterate on ranking/filters and LLM messaging tone.

4) Scale & harden (12+ weeks):
   - Add embedding-based retrieval, privacy-preserving storage (encrypted at rest), autoscaling inference, and feature-store operationalization. Integrate with district systems and plan support/ops.

Estimated total to a robust MVP: ~4-6 months (discovery → pilot) depending on data quality and stakeholder availability.

SyntaxError: unmatched ')' (284825946.py, line 3)

## Data & privacy considerations

- Only ingest anonymized or consented student identifiers for profiling; prefer cohort-level personalization when possible.
- Store borrow history in encrypted storage; minimize PII in logs and sampled data used for model training.
- Allow librarians and guardians to opt-out and provide an accessibility path for students without accounts.
- Ensure LLM outputs are filtered for age-appropriateness and do not reveal private data.

## Proof-of-concept: TF-IDF content-based recommender

This runnable POC demonstrates how we can use the electronic catalog and borrow history to recommend books. It is intentionally simple (no embeddings, no LLMs) and designed to be easy to run during the prototype phase. The notebook constructs a small synthetic catalog and checkout history, builds TF-IDF vectors from titles+descriptions, derives student profiles from past borrows, and computes top-k recommendations via cosine similarity.

In [ ]:
# Minimal requirements for the POC (install locally if needed):

%pip install pandas scikit-learn numpy


Note: you may need to restart the kernel to use updated packages.


In [5]:
# Create a small synthetic catalog and borrow history
import pandas as pd
import re
import math
import numpy as np

# tiny tokenizer and TF-IDF implementation to avoid heavyweight sklearn/scipy imports
def tokenize(text):
    text = (text or '').lower()
    # simple word tokenizer; keep words of length >=2
    return [t for t in re.findall(r"\w+", text) if len(t) > 1]
,

''

In [7]:
# Build a small synthetic borrow history (student_id, book_id, ts)
import pandas as pd
from collections import defaultdict
import numpy as np

borrow_history = pd.DataFrame([
    {"student_id": "S1", "book_id": 1, "ts": "2024-03-01"},
    {"student_id": "S1", "book_id": 4, "ts": "2024-03-15"},
    {"student_id": "S2", "book_id": 2, "ts": "2024-02-05"},
    {"student_id": "S2", "book_id": 4, "ts": "2024-02-20"},
    {"student_id": "S3", "book_id": 3, "ts": "2024-01-10"},
    {"student_id": "S3", "book_id": 2, "ts": "2024-02-10"},
    {"student_id": "S4", "book_id": 5, "ts": "2024-03-03"},
])

# Ensure tfidf (matrix) and catalog are present
try:
    tfidf
    catalog
except NameError:
    raise RuntimeError('TF-IDF matrix or catalog not found. Run the catalog cell first.')

# Create train/test split: hold out the most recent borrow per student with >=2 borrows
borrow_history['ts'] = pd.to_datetime(borrow_history['ts'])
train_rows = []
holdout_rows = []
for sid, group in borrow_history.groupby('student_id'):
    rows = group.sort_values('ts')
    if len(rows) >= 2:
        holdout_rows.append(rows.iloc[-1].to_dict())
        for r in rows.iloc[:-1].to_dict('records'):
            train_rows.append(r)
    else:
        for r in rows.to_dict('records'):
            train_rows.append(r)

train = pd.DataFrame(train_rows) if train_rows else pd.DataFrame(columns=['student_id','book_id','ts'])
holdout = pd.DataFrame(holdout_rows) if holdout_rows else pd.DataFrame(columns=['student_id','book_id','ts'])

# Helper: cosine similarity between a single vector and matrix using numpy
def cosine_sim(a, B):
    # a: (d,), B: (n,d) -> (n,)
    a_norm = np.linalg.norm(a)
    if a_norm == 0:
        return np.zeros(B.shape[0])
    b_norms = np.linalg.norm(B, axis=1)
    dots = B.dot(a)
    denom = a_norm * b_norms
    sim = np.zeros_like(dots)
    nz = denom > 0
    sim[nz] = dots[nz] / denom[nz]
    return sim

# Build student profile vectors (mean TF-IDF of books they've borrowed in train)
student_vecs = {}
for sid, g in train.groupby('student_id'):
    idxs = [book_index[b] for b in g['book_id'] if b in book_index]
    if not idxs:
        continue
    vecs = tfidf[idxs]
    avg = np.asarray(vecs.mean(axis=0)).ravel()
    student_vecs[sid] = avg

print('train size:', len(train), 'holdout size:', len(holdout))

# Recommendation function
def recommend_for_student(student_id, top_k=3):
    if student_id not in student_vecs:
        return []
    profile = student_vecs[student_id]
    sims = cosine_sim(profile, tfidf)
    # exclude already seen books in train
    seen = set(train[train['student_id'] == student_id]['book_id'].tolist())
    candidates = []
    for i, bid in enumerate(catalog['book_id']):
        if bid in seen:
            continue
        candidates.append((bid, sims[i]))
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

# Show recommendations for students
students = sorted(set(train['student_id'].tolist() + holdout['student_id'].tolist()))
for s in students:
    print('\nStudent', s, 'recommendations:')
    recs = recommend_for_student(s, top_k=3)
    for bid, score in recs:
        row = catalog[catalog['book_id'] == bid].iloc[0]
        print(f"  {bid}: {row['title']} ({score:.3f})")

# Simple evaluation: precision@k
def precision_at_k(k=3):
    if holdout.empty:
        return None
    hits = 0
    total = 0
    for _, row in holdout.iterrows():
        sid = row['student_id']
        true_bid = row['book_id']
        recs = [r[0] for r in recommend_for_student(sid, top_k=k)]
        total += 1
        if true_bid in recs:
            hits += 1
    return hits / total if total else None

for k in (1,3):
    print(f'Precision@{k}:', precision_at_k(k))


RuntimeError: TF-IDF matrix or catalog not found. Run the catalog cell first.

## Embedding-based POC and interactive preview

The TF-IDF POC is a quick prototype; for production-grade semantic retrieval we prefer embedding-based retrieval. Below we attempt to build an embedding index using `sentence-transformers` + FAISS. This cell is guarded: if the heavy dependencies are not available the notebook will fall back to the TF-IDF POC above.

The next code cell builds the index and exposes a small Gradio-based preview for librarians to query and preview recommendations and short LLM-generated blurbs (LLM calls are also guarded).

In [ ]:
# Embedding-based retrieval + Gradio preview (guarded)
import importlib

_HAS_TRANSFORMERS = importlib.util.find_spec('sentence_transformers') is not None
_HAS_FAISS = importlib.util.find_spec('faiss') is not None or importlib.util.find_spec('faiss_cpu') is not None
_HAS_GRADIO = importlib.util.find_spec('gradio') is not None

use_fallback = False
if not (_HAS_TRANSFORMERS and _HAS_FAISS):
    print('sentence-transformers or faiss not available; falling back to TF-IDF POC. To enable embeddings install: pip install sentence-transformers faiss-cpu')
    use_fallback = True

# Build embeddings or fallback
embeddings = None
index = None
if not use_fallback:
    try:
        from sentence_transformers import SentenceTransformer
        import faiss
        model = SentenceTransformer('all-MiniLM-L6-v2')
        texts = catalog['text'].tolist()
        embeddings = model.encode(texts, convert_to_numpy=True)
        dim = embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)
        # normalize for cosine similarity as inner product
        faiss.normalize_L2(embeddings)
        index.add(embeddings)
        print('Built FAISS index with', index.ntotal, 'items')
    except Exception as e:
        print('Failed to build embedding index:', e)
        use_fallback = True

# Small guarded LLM blurb generator (uses Hermes3 if available)
def generate_blurb(book_row):
    # book_row: a pandas Series for catalog row
    try:
        # try importing our local Hermes3 client if present
        from app import gradio_main
        # guarded call - demo fallback
        if hasattr(gradio_main, 'synthesize_from_hits'):
            return gradio_main.synthesize_from_hits([book_row['title'], book_row['description']])
    except Exception:
        pass
    # fallback short blurb
    return f"A great pick: {book_row['title']} — {book_row['description'][:140]}..."

# Recommendation wrapper
import numpy as np

def recommend_by_student(student_id=None, query_text=None, top_k=5):
    # If embeddings available and not fallback, use FAISS
    if not use_fallback and embeddings is not None and index is not None:
        if student_id and student_id in student_vecs:
            # attempt to use student's averaged embedding if model available
            try:
                # compute student embedding via averaging book embeddings for seen books
                seen = train[train['student_id'] == student_id]['book_id'].tolist()
                seen_idxs = [book_index[b] for b in seen if b in book_index]
                if seen_idxs:
                    student_emb = embeddings[seen_idxs].mean(axis=0)
                    student_emb = student_emb / np.linalg.norm(student_emb)
                else:
                    student_emb = None
            except Exception:
                student_emb = None
        else:
            student_emb = None

        if query_text and not student_emb:
            try:
                model = SentenceTransformer('all-MiniLM-L6-v2')
                q_emb = model.encode([query_text], convert_to_numpy=True)
                faiss.normalize_L2(q_emb)
                D, I = index.search(q_emb, top_k)
                results = [(catalog.iloc[i]['book_id'], float(D[0,j]), catalog.iloc[i]) for j,i in enumerate(I[0])]
                return results
        
        if student_emb is not None:
            q = student_emb.reshape(1, -1)
            faiss.normalize_L2(q)
            D, I = index.search(q, top_k)
            results = [(catalog.iloc[i]['book_id'], float(D[0,j]), catalog.iloc[i]) for j,i in enumerate(I[0])]
            return results

    # Fallback: use TF-IDF recommend_for_student or query similarity
    if student_id:
        recs = recommend_for_student(student_id, top_k=top_k)
        return [(catalog[catalog['book_id'] == bid].iloc[0]['book_id'], score, catalog[catalog['book_id'] == bid].iloc[0]) for bid, score in recs]
    if query_text:
        # simple TF-IDF similarity by token overlap
        q_toks = tokenize(query_text)
        q_vec = np.zeros((tfidf.shape[1],), dtype=float)
        from collections import Counter
        c = Counter(q_toks)
        max_tf = max(c.values()) if c else 1
        for t, cnt in c.items():
            if t in vocab:
                idx = vocab.index(t)
                q_vec[idx] = (cnt / max_tf)
        sims = cosine_sim(q_vec, tfidf)
        idxs = np.argsort(sims)[::-1][:top_k]
        return [(catalog.iloc[i]['book_id'], float(sims[i]), catalog.iloc[i]) for i in idxs]

    return []

# Optional: launch Gradio preview if available
if _HAS_GRADIO:
    import gradio as gr

    def gradio_query(student_id, query_text):
        res = recommend_by_student(student_id=student_id or None, query_text=query_text or None, top_k=5)
        out = []
        for bid, score, row in res:
            out.append({'book_id': bid, 'title': row['title'], 'score': round(score, 3), 'blurb': generate_blurb(row)})
        return out

    iface = gr.Interface(
        fn=gradio_query,
        inputs=[gr.Textbox(label='Student ID (optional)'), gr.Textbox(label='Query / Interests (optional)')],
        outputs=gr.JSON(label='Recommendations'),
        title='Librarian Recommendation Preview (POC)'
    )
    print('To run Gradio preview in the notebook, call: iface.launch(share=False)')
else:
    print('Gradio not installed. Install with `pip install gradio` to launch the preview UI.')


SyntaxError: expected 'except' or 'finally' block (3977646610.py, line 78)

In [ ]:
# Small slide deck generator: pptx if available, else markdown fallback
import importlib
from pathlib import Path

if importlib.util.find_spec('pptx'):
    from pptx import Presentation
    prs = Presentation()
    # Title slide
    s = prs.slides.add_slide(prs.slide_layouts[0])
    title = s.shapes.title
    subtitle = s.placeholders[1]
    title.text = "Student Reading Support System"
    subtitle.text = "Proposal summary generated from the notebook"

    # Slide: Problem & Goal
    s = prs.slides.add_slide(prs.slide_layouts[1])
    s.shapes.title.text = "Problem & Goal"
    s.placeholders[1].text = (
        "Librarians need a data-driven system to surface personalized book recommendations and run reading campaigns. "
        "Goal: increase student reading via personalized, age-appropriate suggestions and librarian tooling."
    )

    # Slide: Architecture
    s = prs.slides.add_slide(prs.slide_layouts[1])
    s.shapes.title.text = "Architecture (high-level)"
    s.placeholders[1].text = (
        "Catalog + Checkout → ETL → Feature Store / Vector DB → Recommender → Student UI & Librarian Console. "
        "LLMs generate blurbs and synthesize librarian feedback."
    )

    # Slide: Timeline
    s = prs.slides.add_slide(prs.slide_layouts[1])
    s.shapes.title.text = "Timeline"
    s.placeholders[1].text = "Discovery (2-3w), POC (4-6w), Pilot (8-12w), Scale (12+w)"

    # Slide: Next steps
    s = prs.slides.add_slide(prs.slide_layouts[1])
    s.shapes.title.text = "Next steps"
    s.placeholders[1].text = "Run pilot, add embeddings & A/B tests, integrate with district systems, monitor fairness & privacy."

    out = Path('proposal_student_reading_system_deck.pptx')
    prs.save(out)
    print('Wrote PPTX to', out)
else:
    out = Path('proposal_student_reading_system_summary.md')
    out.write_text('\n'.join([
        '# Student Reading Support System',
        '',
        '## Problem & Goal',
        "Librarians need a data-driven system to surface personalized book recommendations and run reading campaigns.",
        '',
        '## Architecture',
        'Catalog + Checkout → ETL → Feature Store / Vector DB → Recommender → Student UI & Librarian Console',
        '',
        '## Timeline',
        'Discovery (2-3w), POC (4-6w), Pilot (8-12w), Scale (12+w)',
        '',
        '## Next steps',
        'Run pilot, add embeddings & A/B tests, integrate with district systems, monitor fairness & privacy.'
    ]))
    print('Wrote markdown summary to', out)


Wrote markdown summary to proposal_student_reading_system_summary.md


In [ ]:
# Diagnostics: detect availability of optional packages
import importlib
pkgs = ['sentence_transformers','faiss','faiss_cpu','gradio','pptx']
found = {p: bool(importlib.util.find_spec(p)) for p in pkgs}
print('Package availability:', found)

# Print short instructions based on availability
if not found['sentence_transformers']:
    print('\nTo enable embeddings run: pip install sentence-transformers')
if not (found['faiss'] or found['faiss_cpu']):
    print('To enable faiss run: pip install faiss-cpu  # or faiss-gpu where appropriate')
if not found['gradio']:
    print('To enable interactive UI run: pip install gradio')
if not found['pptx']:
    print('To enable PPTX generation run: pip install python-pptx')


Package availability: {'sentence_transformers': False, 'faiss': False, 'faiss_cpu': False, 'gradio': False, 'pptx': False}

To enable embeddings run: pip install sentence-transformers
To enable faiss run: pip install faiss-cpu  # or faiss-gpu where appropriate
To enable interactive UI run: pip install gradio
To enable PPTX generation run: pip install python-pptx
